## Setup and Environment

In [1]:
# Import required libraries
import os
import sys
import json
import openai
from dotenv import load_dotenv
from datetime import datetime

# Add src directory to Python path for module imports
sys.path.append(os.path.abspath('../src'))

# Load environment variables
load_dotenv('../.env')

print("Libraries loaded!")
print(" Environment variables loaded")
print(" Source directory added to Python path:", os.path.abspath('../src'))

Libraries loaded!
 Environment variables loaded
 Source directory added to Python path: c:\projects\TRACE_Transactional_Risk_Analysis_-_Compliance_Engine\src


In [2]:
# OpenAI Setup for Vocareum
import openai

# Option 1: Use the helper function from src package (recommended)
# Uncomment this after implementing foundation_sar.py:
# from src import create_vocareum_openai_client
# client = create_vocareum_openai_client()

# Option 2: Manual setup (for early development)
openai_api_key = os.getenv('OPENAI_API_KEY')

if not openai_api_key:
    print(" WARNING: No OpenAI API key found!")
    print("Please set OPENAI_API_KEY in your .env file")
    print("Get your Vocareum OpenAI API key from 'Cloud Resources' in your workspace")
else:
    # Vocareum requires routing through their servers
    client = openai.OpenAI(
        base_url="https://openai.vocareum.com/v1",
        api_key=openai_api_key
    )
    print(" OpenAI client initialized with Vocareum routing")
    print(f" API key: {openai_api_key[:8]}...{openai_api_key[-4:]}")
    print(" Base URL: https://openai.vocareum.com/v1")
    print("\n Tip: After implementing foundation_sar.py, you can use:")
    print("   from src import create_vocareum_openai_client")
    print("   client = create_vocareum_openai_client()")

 OpenAI client initialized with Vocareum routing
 API key: voc-5914...0110
 Base URL: https://openai.vocareum.com/v1

 Tip: After implementing foundation_sar.py, you can use:
   from src import create_vocareum_openai_client
   client = create_vocareum_openai_client()


## Phase 1 Review: Load Foundation Components

Before building agents, let's ensure your foundation components are working.

In [3]:
# TODO: Import your implemented foundation components
# Uncomment and modify these imports once you've implemented foundation_sar.py

from foundation_sar import (
    CustomerData,
    AccountData,
    TransactionData,
    CaseData,
    RiskAnalystOutput,
    ComplianceOfficerOutput,
    ExplainabilityLogger,
    DataLoader,
    load_csv_data
)

print(" TODO: Import foundation components after implementing foundation_sar.py")
print(" Once imported, you can create sample cases for agent testing")
print("Foundation components imported successfully!")

 TODO: Import foundation components after implementing foundation_sar.py
 Once imported, you can create sample cases for agent testing
Foundation components imported successfully!


In [4]:
# Test foundation components - Load CSV data and create a sample case

logger = ExplainabilityLogger("../outputs/audit_logs/agent_development.jsonl")
loader = DataLoader(logger)

# Load CSV data with correct dtype for ssn_last_4
import pandas as pd
customers_df = pd.read_csv("../data/customers.csv", dtype={'ssn_last_4': str})
accounts_df = pd.read_csv("../data/accounts.csv")
transactions_df = pd.read_csv("../data/transactions.csv")

print(f"Loaded {len(customers_df)} customers, {len(accounts_df)} accounts, {len(transactions_df)} transactions")

# Find a customer with transactions for testing
# We'll look for a customer with high transaction count
txn_per_account = transactions_df.groupby('account_id').size()
high_txn_accounts = txn_per_account[txn_per_account >= 10].index.tolist()

# Find customer with one of these high-transaction accounts
sample_customer_id = None
for account_id in high_txn_accounts:
    account_row = accounts_df[accounts_df['account_id'] == account_id]
    if not account_row.empty:
        sample_customer_id = account_row.iloc[0]['customer_id']
        break

if sample_customer_id is None:
    # Fallback to first customer with any transactions
    for idx, row in accounts_df.iterrows():
        account_id = row['account_id']
        if len(transactions_df[transactions_df['account_id'] == account_id]) > 0:
            sample_customer_id = row['customer_id']
            break

# Get customer data
sample_customer = customers_df[customers_df['customer_id'] == sample_customer_id].iloc[0].to_dict()

# Get all accounts for this customer
sample_accounts = accounts_df[accounts_df['customer_id'] == sample_customer_id].to_dict('records')

# Get all transactions for this customer's accounts
account_ids = [acc['account_id'] for acc in sample_accounts]
sample_transactions = transactions_df[transactions_df['account_id'].isin(account_ids)].to_dict('records')

# Handle NaN values in optional fields
for field in ['phone', 'occupation']:
    if pd.isna(sample_customer.get(field)):
        sample_customer[field] = ''
if pd.isna(sample_customer.get('annual_income')):
    sample_customer['annual_income'] = 0

# Handle NaN in transactions
for txn in sample_transactions:
    if pd.isna(txn.get('counterparty')):
        txn['counterparty'] = ''
    if pd.isna(txn.get('location')):
        txn['location'] = ''

print(f"\nSelected customer: {sample_customer['name']} ({sample_customer['customer_id']})")
print(f"  Accounts: {len(sample_accounts)}")
print(f"  Transactions: {len(sample_transactions)}")

sample_case = loader.create_case_from_data(sample_customer, sample_accounts, sample_transactions)

print(f"\nSample case created: {sample_case.case_id}")
print(f"  Customer: {sample_case.customer.name} ({sample_case.customer.customer_id})")
print(f"  Accounts: {len(sample_case.accounts)}")
print(f"  Transactions: {len(sample_case.transactions)}")
print(f"  Risk Rating: {sample_case.customer.risk_rating}")

Loaded 150 customers, 178 accounts, 4268 transactions

Selected customer: Renee Blair (CUST_0002)
  Accounts: 1
  Transactions: 11

Sample case created: 6c2e4bc6-1dfa-4f4a-8e78-2c22015d6861
  Customer: Renee Blair (CUST_0002)
  Accounts: 1
  Transactions: 11
  Risk Rating: Low


## Phase 2: Risk Analyst Agent Development

The Risk Analyst Agent uses **Chain-of-Thought prompting** to systematically analyze suspicious activity patterns.

### Understanding Chain-of-Thought Prompting

Chain-of-Thought (CoT) prompting guides AI models through step-by-step reasoning:

1. **Explicit Steps**: Break complex reasoning into clear phases
2. **Sequential Logic**: Each step builds on previous ones
3. **Domain Expertise**: Frame AI as subject matter expert
4. **Structured Output**: Guide toward specific response format

In [5]:
# Examine the Chain-of-Thought prompt from the implemented Risk Analyst Agent
from risk_analyst_agent import RiskAnalystAgent, create_chain_of_thought_framework, get_classification_categories

# Display the CoT framework
cot_framework = create_chain_of_thought_framework()
print("Chain-of-Thought Analysis Framework:")
for step, description in cot_framework.items():
    print(f"  {step}: {description}")

print("\nClassification Categories:")
categories = get_classification_categories()
for cat, desc in categories.items():
    print(f"  {cat}: {desc}")

# Preview the system prompt
temp_agent = RiskAnalystAgent.__new__(RiskAnalystAgent)
temp_agent.__init__(client, logger)
print(f"\nSystem prompt length: {len(temp_agent.system_prompt)} chars")
print(f"System prompt preview:\n{temp_agent.system_prompt[:300]}...")

Chain-of-Thought Analysis Framework:
  step_1: Data Review - Examine all available information
  step_2: Pattern Recognition - Identify suspicious indicators
  step_3: Regulatory Mapping - Connect to known typologies
  step_4: Risk Quantification - Assess severity level
  step_5: Classification Decision - Determine final category

Classification Categories:
  Structuring: Transactions designed to avoid reporting thresholds
  Sanctions: Potential sanctions violations or prohibited parties
  Fraud: Fraudulent transactions or identity-related crimes
  Money_Laundering: Complex schemes to obscure illicit fund sources
  Other: Suspicious patterns not fitting standard categories

System prompt length: 1928 chars
System prompt preview:
You are a Senior Financial Crime Risk Analyst with extensive experience in Bank Secrecy Act (BSA) compliance and suspicious activity detection.

Use a Chain-of-Thought, step-by-step reasoning approach to analyze the case data provided.

**Analysis Framework (Ch

In [6]:
# Risk Analyst Agent - Simple Smoke Test

from risk_analyst_agent import RiskAnalystAgent

def simple_risk_analyst_smoke_test():
    """Smoke test: create agent, analyze sample case, verify output structure."""
    print("Risk Analyst Smoke Test")
    print("=" * 40)
    
    # Check if sample_case exists
    if 'sample_case' not in globals():
        print("  SKIPPED: Run the previous cell to create sample_case first")
        return None
    
    try:
        agent = RiskAnalystAgent(client, logger, model="gpt-4o-mini")
        print(f"  Agent initialized (model: {agent.model})")
        
        # Run analysis on the sample case built earlier
        result = agent.analyze_case(sample_case)
        
        # Verify output structure
        assert hasattr(result, 'classification'), "Missing classification"
        assert hasattr(result, 'confidence_score'), "Missing confidence_score"
        assert hasattr(result, 'reasoning'), "Missing reasoning"
        assert hasattr(result, 'key_indicators'), "Missing key_indicators"
        assert hasattr(result, 'risk_level'), "Missing risk_level"
        assert 0.0 <= result.confidence_score <= 1.0, "Confidence out of range"
        
        print(f"  Classification: {result.classification}")
        print(f"  Confidence: {result.confidence_score}")
        print(f"  Risk Level: {result.risk_level}")
        print(f"  Key Indicators: {result.key_indicators}")
        print(f"  Reasoning: {result.reasoning[:200]}...")
        print("\n  SUCCESS: Risk Analyst smoke test passed!")
        return result
        
    except Exception as e:
        print(f"\n  FAILED: {e}")
        import traceback
        traceback.print_exc()
        return None

risk_analysis_result = simple_risk_analyst_smoke_test()

Risk Analyst Smoke Test
  Agent initialized (model: gpt-4o-mini)
  Classification: Other
  Confidence: 0.6
  Risk Level: Medium
  Key Indicators: ['High average monthly balance compared to current balance', 'No suspicious transaction patterns']
  Reasoning: The customer has a low risk rating and a stable income. Transactions are consistent with their profile, with no structuring or unusual patterns. However, the average monthly balance is significantly h...

  SUCCESS: Risk Analyst smoke test passed!


###  Risk Analyst Testing Framework

In [7]:
# Comprehensive Risk Analyst Testing - Run Pre-Built Test Suite

import sys
import os

project_root = os.path.abspath('..')
tests_path = os.path.join(project_root, 'tests')
if tests_path not in sys.path:
    sys.path.insert(0, tests_path)

print(f"Tests directory: {tests_path}")

# Preview available tests
try:
    from test_risk_analyst import TestRiskAnalystAgent
    
    test_methods = [m for m in dir(TestRiskAnalystAgent) if m.startswith('test_')]
    print(f"\nAvailable Risk Analyst Tests ({len(test_methods)}):")
    for method_name in test_methods:
        method = getattr(TestRiskAnalystAgent, method_name)
        doc = method.__doc__ or method_name.replace('_', ' ').title()
        print(f"  - {doc}")
except Exception as e:
    print(f"Could not load test suite: {e}")

# Run the comprehensive test suite
try:
    import pytest
    
    print("\nRunning Risk Analyst comprehensive test suite...")
    result = pytest.main([
        f"{tests_path}/test_risk_analyst.py",
        "-v",
        "--tb=short"
    ])
    
    if result == 0:
        print("\nAll Risk Analyst tests passed!")
    else:
        print("\nSome tests failed. Review output above for details.")
except ImportError:
    print("pytest not installed. Run: pip install pytest")

Tests directory: c:\projects\TRACE_Transactional_Risk_Analysis_-_Compliance_Engine\tests
⚠️  Risk Analyst Agent not yet implemented - tests will be skipped
💡 Implement the RiskAnalystAgent class in src/risk_analyst_agent.py to run these tests

Available Risk Analyst Tests (10):
  - Test RiskAnalystAgent initializes properly
  - Test handling of invalid JSON response
  - Test successful case analysis with valid response
  - Test OpenAI API call uses correct parameters
  - Test handling of empty LLM response
  - Test JSON extraction from code blocks
  - Test JSON extraction from plain text response
  - Test account formatting for prompts
  - Test transaction formatting for prompts
  - Test system prompt contains required elements

Running Risk Analyst comprehensive test suite...
============================= test session starts =============================
platform win32 -- Python 3.11.13, pytest-9.0.2, pluggy-1.6.0 -- c:\projects\TRACE_Transactional_Risk_Analysis_-_Compliance_Engine\.v

## Phase 3: Compliance Officer Agent Development

The Compliance Officer Agent uses **ReACT prompting** to generate regulatory-compliant SAR narratives.

###  Understanding ReACT Prompting

ReACT (Reasoning + Action) prompting separates thinking and doing:

1. **Reasoning Phase**: Analyze situation and plan approach
2. **Action Phase**: Execute specific task with informed decisions
3. **Structured Workflow**: Consistent approach to complex tasks
4. **Regulatory Compliance**: Emphasis on meeting specific requirements

In [8]:
# Examine the ReACT prompt from the implemented Compliance Officer Agent
from compliance_officer_agent import ComplianceOfficerAgent, create_react_framework, get_regulatory_requirements

# Display the ReACT framework
react_framework = create_react_framework()
print("ReACT Framework:")
print("  REASONING Phase:")
for step in react_framework['reasoning_phase']:
    print(f"    - {step}")
print("  ACTION Phase:")
for step in react_framework['action_phase']:
    print(f"    - {step}")

print("\nRegulatory Requirements:")
reqs = get_regulatory_requirements()
print(f"  Word limit: {reqs['word_limit']}")
print(f"  Required elements: {reqs['required_elements']}")
print(f"  Citations: {reqs['citations']}")

# Preview the system prompt
temp_co = ComplianceOfficerAgent(client, logger)
print(f"\nSystem prompt length: {len(temp_co.system_prompt)} chars")
print(f"System prompt preview:\n{temp_co.system_prompt[:300]}...")

ReACT Framework:
  REASONING Phase:
    - Review risk analysis findings
    - Assess regulatory requirements
    - Identify compliance elements
    - Plan narrative structure
  ACTION Phase:
    - Draft concise narrative
    - Include specific details
    - Reference activity patterns
    - Use regulatory language

Regulatory Requirements:
  Word limit: 120
  Required elements: ['Customer identification', 'Suspicious activity description', 'Transaction amounts and dates', 'Why activity is suspicious']
  Citations: ['31 CFR 1020.320 (BSA)', '12 CFR 21.11 (SAR Filing)', 'FinCEN SAR Instructions']

System prompt length: 1931 chars
System prompt preview:
You are a Senior Compliance Officer specializing in BSA/AML regulatory compliance and SAR narrative generation for FinCEN submission.

Use the ReACT (REASONING + Action) framework to generate regulatory-compliant SAR narratives.

**REASONING Phase:**
1. Review the Risk Analyst's findings including c...


In [9]:
# Compliance Officer Agent - Simple Smoke Test

from compliance_officer_agent import ComplianceOfficerAgent

def simple_compliance_officer_smoke_test():
    """Smoke test: create agent, generate narrative, verify output structure."""
    print("Compliance Officer Smoke Test")
    print("=" * 40)
    
    # Check if required variables exist
    if 'sample_case' not in globals():
        print("  SKIPPED: Run the data loading cell to create sample_case first")
        return None
    
    if 'risk_analysis_result' not in globals() or risk_analysis_result is None:
        print("  SKIPPED: Risk Analyst smoke test must pass first")
        return None
    
    try:
        agent = ComplianceOfficerAgent(client, logger, model="gpt-4o-mini")
        print(f"  Agent initialized (model: {agent.model})")
        
        # Generate compliance narrative using sample case + risk analysis
        result = agent.generate_compliance_narrative(sample_case, risk_analysis_result)
        
        # Verify output structure
        assert hasattr(result, 'narrative'), "Missing narrative"
        assert hasattr(result, 'narrative_reasoning'), "Missing narrative_reasoning"
        assert hasattr(result, 'regulatory_citations'), "Missing regulatory_citations"
        assert hasattr(result, 'completeness_check'), "Missing completeness_check"
        
        word_count = len(result.narrative.split())
        assert word_count <= 120, f"Narrative exceeds 120 words ({word_count})"
        
        print(f"  Narrative ({word_count} words): {result.narrative[:150]}...")
        print(f"  Reasoning: {result.narrative_reasoning[:150]}...")
        print(f"  Citations: {result.regulatory_citations}")
        print(f"  Completeness: {result.completeness_check}")
        print(f"\n  SUCCESS: Compliance Officer smoke test passed!")
        return result
        
    except Exception as e:
        print(f"\n  FAILED: {e}")
        import traceback
        traceback.print_exc()
        return None

compliance_result = simple_compliance_officer_smoke_test()

Compliance Officer Smoke Test
  Agent initialized (model: gpt-4o-mini)
  Narrative (88 words): Renee Blair (CUST_0002) has exhibited unusual banking behavior with a significant average monthly balance of $4,000, contrasting with a current balanc...
  Reasoning: The narrative highlights the customer's identity, summarizes key transactions, and explains the suspicious nature of the activity based on the average...
  Citations: ['31 CFR 1020.320', '31 USC 5324']
  Completeness: True

  SUCCESS: Compliance Officer smoke test passed!


### Compliance Officer Testing Framework

In [10]:
# Comprehensive Compliance Officer Testing - Run Pre-Built Test Suite

import sys
import os

project_root = os.path.abspath('..')
tests_path = os.path.join(project_root, 'tests')
if tests_path not in sys.path:
    sys.path.insert(0, tests_path)

# Preview available tests
try:
    from test_compliance_officer import TestComplianceOfficerAgent
    
    test_methods = [m for m in dir(TestComplianceOfficerAgent) if m.startswith('test_')]
    print(f"Available Compliance Officer Tests ({len(test_methods)}):")
    for method_name in test_methods:
        method = getattr(TestComplianceOfficerAgent, method_name)
        doc = method.__doc__ or method_name.replace('_', ' ').title()
        print(f"  - {doc}")
except Exception as e:
    print(f"Could not load test suite: {e}")

# Run the comprehensive test suite
try:
    import pytest
    
    print("\nRunning Compliance Officer comprehensive test suite...")
    result = pytest.main([
        f"{tests_path}/test_compliance_officer.py",
        "-v",
        "--tb=short"
    ])
    
    if result == 0:
        print("\nAll Compliance Officer tests passed!")
    else:
        print("\nSome tests failed. Review output above for details.")
except ImportError:
    print("pytest not installed. Run: pip install pytest")

Available Compliance Officer Tests (10):
  - Test ComplianceOfficerAgent initializes properly
  - Test OpenAI API call uses correct parameters
  - Test handling of empty LLM response
  - Test JSON extraction from code blocks
  - Test JSON extraction from plain text response
  - Test transaction formatting for compliance narratives
  - Test successful narrative generation with valid response
  - Test handling of invalid JSON response
  - Test narrative word count validation (120 word limit)
  - Test system prompt contains required ReACT elements

Running Compliance Officer comprehensive test suite...
============================= test session starts =============================
platform win32 -- Python 3.11.13, pytest-9.0.2, pluggy-1.6.0 -- c:\projects\TRACE_Transactional_Risk_Analysis_-_Compliance_Engine\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: c:\projects\TRACE_Transactional_Risk_Analysis_-_Compliance_Engine
configfile: pyproject.toml
plugins: anyio-4.12.1
collecting

In [11]:
# Complete Agent Testing - Run both test suites together

import pytest

print("Complete Agent Testing")
print("=" * 50)

print("\nTIER 1: Smoke Test Results")
print(f"  Risk Analyst:       {'PASSED' if risk_analysis_result else 'FAILED'}")
print(f"  Compliance Officer: {'PASSED' if compliance_result else 'FAILED'}")

print("\nTIER 2: Comprehensive Test Suites")

# Quick validation of both agents
risk_result = pytest.main([
    f"{tests_path}/test_risk_analyst.py::TestRiskAnalystAgent::test_agent_initialization",
    f"{tests_path}/test_risk_analyst.py::TestRiskAnalystAgent::test_analyze_case_success",
    "-v", "--tb=short"
])

compliance_test_result = pytest.main([
    f"{tests_path}/test_compliance_officer.py::TestComplianceOfficerAgent::test_agent_initialization",
    f"{tests_path}/test_compliance_officer.py::TestComplianceOfficerAgent::test_generate_compliance_narrative_success",
    "-v", "--tb=short"
])

if risk_result == 0 and compliance_test_result == 0:
    print("\nBoth agents passing key tests! Ready for full test suite and workflow integration.")
else:
    if risk_result != 0:
        print("\nRisk Analyst needs fixes.")
    if compliance_test_result != 0:
        print("\nCompliance Officer needs fixes.")

Complete Agent Testing

TIER 1: Smoke Test Results
  Risk Analyst:       PASSED
  Compliance Officer: PASSED

TIER 2: Comprehensive Test Suites
============================= test session starts =============================
platform win32 -- Python 3.11.13, pytest-9.0.2, pluggy-1.6.0 -- c:\projects\TRACE_Transactional_Risk_Analysis_-_Compliance_Engine\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: c:\projects\TRACE_Transactional_Risk_Analysis_-_Compliance_Engine
configfile: pyproject.toml
plugins: anyio-4.12.1
collecting ... collected 2 items

..\tests\test_risk_analyst.py::TestRiskAnalystAgent::test_agent_initialization PASSED [ 50%]
..\tests\test_risk_analyst.py::TestRiskAnalystAgent::test_analyze_case_success PASSED [100%]

============================== warnings summary ===============================
..\.venv\Lib\site-packages\_pytest\config\__init__.py:1303
  c:\projects\TRACE_Transactional_Risk_Analysis_-_Compliance_Engine\.venv\Lib\site-packages\_pytest\config\__init

## Phase 4 Preview: Agent Integration

Once both agents are working, you'll integrate them into a complete workflow.

In [12]:
# TODO: Preview of integrated workflow
# This will be fully implemented in the next notebook

def preview_integrated_workflow():
    """Preview of how agents will work together"""
    
    workflow_steps = [
        "1.  Load and validate case data",
        "2.  Risk Analyst performs Chain-of-Thought analysis",
        "3.  Human review and approval gate",
        "4.  Compliance Officer generates ReACT narrative (if approved)",
        "5.  Generate complete SAR document",
        "6.  Log audit trail and efficiency metrics"
    ]
    
    print(" Integrated SAR Processing Workflow:")
    for step in workflow_steps:
        print(step)
    
    print("\n Key Benefits:")
    print("• Two-stage processing reduces AI costs")
    print("• Human oversight ensures regulatory compliance")
    print("• Complete audit trails for examination")
    print("• Standardized analytical approaches")

preview_integrated_workflow()

 Integrated SAR Processing Workflow:
1.  Load and validate case data
2.  Risk Analyst performs Chain-of-Thought analysis
3.  Human review and approval gate
4.  Compliance Officer generates ReACT narrative (if approved)
5.  Generate complete SAR document
6.  Log audit trail and efficiency metrics

 Key Benefits:
• Two-stage processing reduces AI costs
• Human oversight ensures regulatory compliance
• Complete audit trails for examination
• Standardized analytical approaches


## Development Checklist - Two-Tier Testing Approach

### Risk Analyst Agent (Phase 2)
- [ ] Implement Chain-of-Thought system prompt
- [ ] Create `analyze_case` method with error handling
- [ ] Add JSON parsing and validation
- [ ] **TIER 1**: Write simple smoke test (verify basic functionality)
- [ ] **TIER 2**: Run comprehensive pre-built test suite (10 comprehensive tests)
- [ ] Fix any issues identified by test suite

### Compliance Officer Agent (Phase 3)  
- [ ] Implement ReACT system prompt
- [ ] Create `generate_compliance_narrative` method
- [ ] Add narrative validation (word count, terminology)
- [ ] **TIER 1**: Write simple smoke test (verify basic functionality)
- [ ] **TIER 2**: Run comprehensive pre-built test suite (10 comprehensive tests)
- [ ] Fix any issues identified by test suite

### Testing Strategy Benefits
- [ ] **Time Savings**: Focus on implementation, not complex test creation
- [ ] **Better Coverage**: Pre-built test suites test edge cases you might miss
- [ ] **Quick Feedback**: Simple smoke tests for rapid development cycles
- [ ] **Professional Validation**: Comprehensive test suites ensure production readiness
- [ ] **Regulatory Compliance**: Built-in checks for SAR requirements

### **Testing Workflow**
1. **Start with Tier 1**: Write simple smoke tests to verify your agents don't crash
2. **Fix basic issues**: Iterate quickly with simple tests during development
3. **Move to Tier 2**: Run comprehensive test suites when basic functionality works
4. **Analyze results**: Use detailed feedback to improve agent performance
5. **Iterate**: Refine prompts and logic based on test results

## Next Steps

1. **Complete Agent Implementation**: Finish both agent classes in the src/ directory
2. **Run Two-Tier Testing**: Start with smoke tests, then comprehensive test suites
3. **Workflow Integration**: Move to the next notebook for complete system integration
4. **Human-in-the-Loop**: Implement decision gates and review processes

## Available Test Suites Summary

**Risk Analyst Test Suite (10 tests):**
- Agent initialization and configuration
- Case analysis with valid JSON responses
- JSON parsing and error handling
- System prompt structure validation
- API call parameter verification
- Helper method functionality
- Edge case handling

**Compliance Officer Test Suite (10 tests):**
- Agent initialization and configuration
- Narrative generation with valid responses
- Word count validation (≤120 words)
- Regulatory citations inclusion
- JSON parsing and error handling
- ReACT prompt structure validation
- API call parameter verification

**Ready to build intelligent agents with professional-grade testing!**